# MINA Voice API para Google Colab\n\nEste notebook levanta una API pequeña para generar voz de MINA usando la GPU de Colab.\n\nPasos rápidos:\n1. En Colab ve a **Entorno de ejecución > Cambiar tipo de entorno de ejecución > GPU**.\n2. Ejecuta las celdas en orden.\n3. Sube tu archivo `user_reference.wav` cuando lo pida.\n4. Copia la URL `https://...trycloudflare.com` que salga al final.\n5. En el `.env` de MINA pon `VOICE_REMOTE_URL=esa_url` y reinicia el bot.\n

In [ ]:
# 1) Instalar dependencias\n!pip -q install chatterbox-tts fastapi uvicorn nest-asyncio python-multipart\n!wget -q -O /content/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64\n!chmod +x /content/cloudflared\n

In [ ]:
# 2) Subir referencia de voz\n# Sube el archivo del proyecto: assets/voice/user_reference.wav\nfrom google.colab import files\nuploaded = files.upload()\n\nimport shutil\nfrom pathlib import Path\n\nif not uploaded:\n    raise RuntimeError('No subiste ningún archivo de referencia.')\n\nfirst_file = next(iter(uploaded.keys()))\nreference_path = Path('/content/user_reference.wav')\nshutil.copy(first_file, reference_path)\nprint('Referencia lista en:', reference_path)\n

In [ ]:
# 3) Cargar modelo y levantar API\nimport os\nimport tempfile\nimport torch\nimport torchaudio as ta\nimport nest_asyncio\nimport uvicorn\nfrom pathlib import Path\nfrom fastapi import FastAPI, Header, HTTPException\nfrom fastapi.responses import FileResponse\nfrom pydantic import BaseModel\n\ntry:\n    import perth\n    if getattr(perth, 'PerthImplicitWatermarker', None) is None:\n        class DummyWatermarker:\n            def apply_watermark(self, wav, sample_rate=None):\n                return wav\n        perth.PerthImplicitWatermarker = DummyWatermarker\nexcept Exception:\n    pass\n\ntry:\n    from chatterbox.mtl_tts import ChatterboxMultilingualTTS\nexcept Exception:\n    ChatterboxMultilingualTTS = None\nfrom chatterbox.tts import ChatterboxTTS\n\nREFERENCE = Path('/content/user_reference.wav')\nif not REFERENCE.exists():\n    raise RuntimeError('Falta /content/user_reference.wav. Ejecuta la celda de subida primero.')\n\nDEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'\nUSE_MULTILINGUAL = ChatterboxMultilingualTTS is not None\nprint('GPU/CPU:', DEVICE)\n\nif USE_MULTILINGUAL:\n    model = ChatterboxMultilingualTTS.from_pretrained(device=DEVICE)\n    model_name = 'ChatterboxMultilingualTTS'\nelse:\n    model = ChatterboxTTS.from_pretrained(device=DEVICE)\n    model.prepare_conditionals(str(REFERENCE), exaggeration=0.75)\n    model_name = 'ChatterboxTTS'\n\nprint('Modelo listo:', model_name, 'sample_rate:', model.sr)\n\nVOICE_TOKEN = os.environ.get('MINA_VOICE_TOKEN', '').strip()\napp = FastAPI(title='MINA Voice API')\n\nclass GenerateRequest(BaseModel):\n    text: str\n\ndef check_auth(authorization: str | None):\n    if not VOICE_TOKEN:\n        return\n    if authorization != f'Bearer {VOICE_TOKEN}':\n        raise HTTPException(status_code=401, detail='Token inválido')\n\n@app.get('/health')\ndef health(authorization: str | None = Header(default=None)):\n    check_auth(authorization)\n    return {'ok': True, 'device': DEVICE, 'model': model_name, 'sample_rate': model.sr}\n\n@app.post('/generate')\ndef generate(req: GenerateRequest, authorization: str | None = Header(default=None)):\n    check_auth(authorization)\n    text = req.text.strip()[:260]\n    if not text:\n        raise HTTPException(status_code=400, detail='Texto vacío')\n    out_path = Path(tempfile.gettempdir()) / f'mina_voice_{abs(hash(text))}_{os.getpid()}.wav'\n    if USE_MULTILINGUAL:\n        wav = model.generate(\n            text,\n            'es',\n            audio_prompt_path=str(REFERENCE),\n            repetition_penalty=1.18,\n            min_p=0.02,\n            top_p=0.92,\n            exaggeration=0.95,\n            cfg_weight=0.60,\n            temperature=0.96,\n        )\n    else:\n        wav = model.generate(\n            text,\n            repetition_penalty=1.12,\n            min_p=0.03,\n            top_p=0.94,\n            exaggeration=0.88,\n            cfg_weight=0.48,\n            temperature=0.90,\n        )\n    ta.save(str(out_path), wav, model.sr)\n    return FileResponse(str(out_path), media_type='audio/wav', filename='mina.wav')\n\nnest_asyncio.apply()\nserver = uvicorn.Server(uvicorn.Config(app, host='127.0.0.1', port=7860, log_level='warning'))\n\nimport threading\nthreading.Thread(target=server.run, daemon=True).start()\nprint('API local lista en http://127.0.0.1:7860')\n

In [ ]:
# 4) Abrir túnel público con Cloudflare\nimport re\nimport subprocess\nimport threading\nimport time\n\nproc = subprocess.Popen(\n    ['/content/cloudflared', 'tunnel', '--url', 'http://127.0.0.1:7860'],\n    stdout=subprocess.PIPE,\n    stderr=subprocess.STDOUT,\n    text=True,\n)\n\npublic_url = None\n\ndef read_logs():\n    global public_url\n    for line in proc.stdout:\n        print(line, end='')\n        match = re.search(r'https://[-a-zA-Z0-9.]+\\.trycloudflare\\.com', line)\n        if match and public_url is None:\n            public_url = match.group(0)\n\nthreading.Thread(target=read_logs, daemon=True).start()\n\nfor _ in range(30):\n    if public_url:\n        break\n    time.sleep(1)\n\nif not public_url:\n    raise RuntimeError('No se pudo obtener la URL pública todavía. Revisa los logs de cloudflared arriba.')\n\nprint('\nCOPIA ESTA URL EN EL .env DE MINA:')\nprint('VOICE_REMOTE_URL=' + public_url)\n